# K-Nearest Neighbors (KNN) Algorithm

This notebook provides a university-level introduction to the K-Nearest Neighbors algorithm, covering both theoretical foundations and practical implementations in Python.

Author: Michał Woźniak

## Technical Setup: Running This Notebook Locally with `uv`

### Prerequisites
- Ensure you have `uv` installed on your system. If not, install it following the instructions at [https://github.com/astral-sh/uv](https://github.com/astral-sh/uv)

### Step 1: Initialize a New Project
Navigate to your working directory and initialize a new Python project:
```bash
# Navigate to your project directory
cd /path/to/your/project

# Initialize uv project (if not already initialized)
uv init

# Or if you want to specify Python version
uv init --python 3.11
```

### Step 2: Install Required Dependencies
Install all necessary packages for this notebook:
```bash
# Install core data science packages
uv add numpy pandas scikit-learn scipy

# Install visualization libraries
uv add matplotlib seaborn

# Install Jupyter notebook
uv add jupyter ipykernel

# Alternative: Install all at once
uv add numpy pandas scikit-learn scipy matplotlib seaborn jupyter ipykernel
```

### Step 3: Activate the Environment and Start Jupyter
```bash
# Activate the virtual environment (uv handles this automatically)
# Start Jupyter notebook
uv run jupyter notebook

# Or use Jupyter Lab
uv run jupyter lab
```

### Step 4: Open This Notebook
Once Jupyter starts, navigate to `lab_2/KNN_Tutorial.ipynb` and open it.

### Required Packages Summary
- **numpy** (≥1.21.0): Numerical computations and array operations
- **pandas** (≥1.3.0): Data manipulation and analysis
- **scikit-learn** (≥1.0.0): Machine learning algorithms and utilities
- **scipy** (≥1.7.0): Scientific computing (K-D tree implementation)
- **matplotlib** (≥3.4.0): Basic plotting and visualization
- **seaborn** (≥0.11.0): Statistical data visualization
- **jupyter**: Jupyter notebook interface
- **ipykernel**: Jupyter kernel for Python

### Troubleshooting
- If you encounter import errors, ensure all packages are installed: `uv pip list`
- To update packages: `uv add --upgrade package_name`
- To see your Python version: `uv run python --version`


# 1. Introduction to K-Nearest Neighbors

## What is K-Nearest Neighbors?

K-Nearest Neighbors (KNN) is a **non-parametric**, **instance-based**, **lazy learning** algorithm used for both classification and regression tasks.

### Key Characteristics:

1. **Non-parametric**: KNN makes no assumptions about the underlying data distribution. It doesn't learn a discriminative function from the training data but memorizes the training dataset instead.

2. **Instance-based (Memory-based) Learning**: The algorithm stores all training instances and uses them directly for prediction, rather than building an explicit model.

3. **Lazy Learning**: KNN doesn't have a traditional "training" phase. All computation is deferred until prediction time, making training instantaneous but prediction potentially slow.

### How KNN Works:

The core idea is beautifully simple:
- **For Classification**: Given a new data point, find the K nearest neighbors in the training data and assign the most common class among them.
- **For Regression**: Given a new data point, find the K nearest neighbors and predict the average (or weighted average) of their values.

### Mathematical Foundation

#### Distance Metrics

The "nearness" of neighbors is determined by a distance metric. Common choices include:

1. **Euclidean Distance** (L2 norm):
   $$d(x, y) = \sqrt{\sum_{i=1}^{n}(x_i - y_i)^2}$$

2. **Manhattan Distance** (L1 norm):
   $$d(x, y) = \sum_{i=1}^{n}|x_i - y_i|$$

3. **Minkowski Distance** (generalized):
   $$d(x, y) = \left(\sum_{i=1}^{n}|x_i - y_i|^p\right)^{1/p}$$
   
   Note: When p=1, it's Manhattan; when p=2, it's Euclidean.

4. **Cosine Distance**: Measures the angle between vectors
   $$d(x, y) = 1 - \frac{x \cdot y}{||x|| \cdot ||y||}$$

#### Prediction Formulas

**Classification** (majority voting):
$$\hat{y} = \text{mode}\{y_1, y_2, ..., y_k\}$$

**Classification with Distance Weighting**:
$$\hat{y} = \arg\max_c \sum_{i=1}^{k} w_i \cdot \mathbb{1}(y_i = c)$$
where $w_i = \frac{1}{d(x, x_i)^2}$ or $w_i = \frac{1}{d(x, x_i)}$

**Regression** (mean):
$$\hat{y} = \frac{1}{k}\sum_{i=1}^{k} y_i$$

**Regression with Distance Weighting**:
$$\hat{y} = \frac{\sum_{i=1}^{k} w_i \cdot y_i}{\sum_{i=1}^{k} w_i}$$


## Key Hyperparameters

### 1. Number of Neighbors (k)
- **Small k**: More sensitive to noise, can lead to overfitting
- **Large k**: Smoother decision boundaries, but can lead to underfitting
- **Rule of thumb**: Start with $k = \sqrt{n}$ where n is the number of training samples
- **Best practice**: Use cross-validation to find optimal k

### 2. Distance Metric
- Choice depends on the nature of your features
- Euclidean works well for continuous features
- Manhattan is robust to outliers
- Cosine is good for text data or when direction matters more than magnitude

### 3. Weighting Scheme
- **Uniform**: All neighbors have equal weight
- **Distance**: Closer neighbors have more influence
  - Weight = 1/distance or 1/distance²

### 4. Algorithm for Finding Neighbors
- **Brute force**: Computes distance to all points (O(n) per query)
- **K-D Tree**: Efficient for low-dimensional data (O(log n) per query)
- **Ball Tree**: Works better for high-dimensional data

## Decision Boundaries

In classification, KNN creates **non-linear decision boundaries**. These boundaries are determined by the regions where different classes dominate based on the k nearest neighbors.

- With k=1, the decision boundary can be very irregular (high variance)
- As k increases, the boundary becomes smoother (lower variance, higher bias)
- The boundary is influenced by the class distribution in the training data


## Advantages and Limitations

### Advantages ✓
1. **Simple to understand and implement**: Intuitive concept based on similarity
2. **No training phase**: Model is "trained" instantly
3. **Non-parametric**: Makes no assumptions about data distribution
4. **Naturally handles multi-class problems**: Works for any number of classes
5. **Effective for small datasets**: Can perform well when data is limited

### Limitations ✗
1. **Computationally expensive predictions**: Must calculate distance to all training points
2. **Memory intensive**: Stores all training data
3. **Sensitive to irrelevant features**: All features equally contribute to distance
4. **Curse of dimensionality**: Performance degrades in high-dimensional spaces
5. **Requires feature scaling**: Features with large ranges dominate the distance calculation
6. **Imbalanced data issues**: Majority class can dominate predictions

## The Curse of Dimensionality

As the number of dimensions increases:
- Data points become increasingly sparse
- All points become approximately equidistant from each other
- The concept of "nearest" neighbor becomes less meaningful
- Computational complexity increases exponentially

**Rule of thumb**: KNN works best when features < 20-30 dimensions. Beyond this, consider dimensionality reduction techniques (PCA, t-SNE, etc.).

## Critical Preprocessing: Feature Scaling

**Why is feature scaling essential for KNN?**

Consider this example:
- Feature 1: Age (range 20-80)
- Feature 2: Income (range 20,000-200,000)

Without scaling, income would dominate the Euclidean distance calculation because it has a much larger range. The algorithm would essentially ignore age!

**Common scaling methods:**

1. **Min-Max Scaling (Normalization)**: Scales to [0, 1]
   $$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

2. **Standardization (Z-score normalization)**: Zero mean, unit variance
   $$x' = \frac{x - \mu}{\sigma}$$

3. **Robust Scaling**: Uses median and IQR (robust to outliers)
   $$x' = \frac{x - \text{median}}{IQR}$$

**Best practice**: Apply the same scaling to both training and test data using parameters learned from training data only!


# 2. From-Scratch Implementation: Classification

In this section, we'll implement KNN from scratch to understand the underlying mechanics. We'll create both a brute-force version and a K-D tree version, then compare their performance.

## 2.1 Brute-Force KNN Implementation

Let's implement the basic KNN algorithm step by step.


In [ ]:
# Import necessary libraries
import time
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Matplotlib configuration for better plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")


In [ ]:
# Load the Iris dataset
from sklearn.datasets import load_iris

# Load data
iris = load_iris()
X = iris.data
y = iris.target

print("Iris Dataset Loaded")
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature names: {iris.feature_names}")
print(f"Target names: {iris.target_names}")
print(f"\nFirst 5 samples:\n{X[:5]}")
print(f"First 5 labels: {y[:5]}")


### Step 1: Implement Distance Calculation Functions


In [ ]:
def euclidean_distance(x1, x2):
    """
    Calculate Euclidean distance between two vectors.

    Parameters:
    -----------
    x1, x2 : array-like
        Input vectors

    Returns:
    --------
    float : Euclidean distance
    """
    return np.sqrt(np.sum((x1 - x2) ** 2))

def manhattan_distance(x1, x2):
    """
    Calculate Manhattan distance between two vectors.

    Parameters:
    -----------
    x1, x2 : array-like
        Input vectors

    Returns:
    --------
    float : Manhattan distance
    """
    return np.sum(np.abs(x1 - x2))

def minkowski_distance(x1, x2, p=3):
    """
    Calculate Minkowski distance between two vectors.

    Parameters:
    -----------
    x1, x2 : array-like
        Input vectors
    p : int
        The order of the norm

    Returns:
    --------
    float : Minkowski distance
    """
    return np.power(np.sum(np.abs(x1 - x2) ** p), 1/p)

# Test the distance functions
point1 = np.array([1, 2, 3])
point2 = np.array([4, 5, 6])

print("Distance between", point1, "and", point2)
print(f"Euclidean distance: {euclidean_distance(point1, point2):.4f}")
print(f"Manhattan distance: {manhattan_distance(point1, point2):.4f}")
print(f"Minkowski distance (p=3): {minkowski_distance(point1, point2, p=3):.4f}")


### Step 2: Implement KNN Classifier from Scratch


In [ ]:
class KNNClassifier:
    """
    K-Nearest Neighbors Classifier implemented from scratch using brute-force approach.
    """

    def __init__(self, k=3, distance_metric='euclidean'):
        """
        Initialize KNN Classifier.

        Parameters:
        -----------
        k : int
            Number of nearest neighbors to consider
        distance_metric : str
            Distance metric to use ('euclidean', 'manhattan', or 'minkowski')
        """
        self.k = k
        self.distance_metric = distance_metric

    def fit(self, X_train, y_train):
        """
        Fit the model (simply store the training data).

        Parameters:
        -----------
        X_train : array-like, shape (n_samples, n_features)
            Training data
        y_train : array-like, shape (n_samples,)
            Target labels
        """
        self.X_train = np.array(X_train)
        self.y_train = np.array(y_train)
        return self

    def _get_distance_function(self):
        """Return the appropriate distance function."""
        if self.distance_metric == 'euclidean':
            return euclidean_distance
        elif self.distance_metric == 'manhattan':
            return manhattan_distance
        elif self.distance_metric == 'minkowski':
            return lambda x1, x2: minkowski_distance(x1, x2, p=3)
        else:
            raise ValueError(f"Unknown distance metric: {self.distance_metric}")

    def predict_single(self, x):
        """
        Predict the class for a single sample.

        Parameters:
        -----------
        x : array-like, shape (n_features,)
            Test sample

        Returns:
        --------
        int : Predicted class label
        """
        # Calculate distances to all training samples
        distance_func = self._get_distance_function()
        distances = [distance_func(x, x_train) for x_train in self.X_train]

        # Get indices of k nearest neighbors
        k_indices = np.argsort(distances)[:self.k]

        # Get labels of k nearest neighbors
        k_nearest_labels = self.y_train[k_indices]

        # Return most common label (majority vote)
        most_common = Counter(k_nearest_labels).most_common(1)
        return most_common[0][0]

    def predict(self, X_test):
        """
        Predict classes for multiple samples.

        Parameters:
        -----------
        X_test : array-like, shape (n_samples, n_features)
            Test data

        Returns:
        --------
        array : Predicted class labels
        """
        X_test = np.array(X_test)
        return np.array([self.predict_single(x) for x in X_test])

    def score(self, X_test, y_test):
        """
        Calculate accuracy score.

        Parameters:
        -----------
        X_test : array-like, shape (n_samples, n_features)
            Test data
        y_test : array-like, shape (n_samples,)
            True labels

        Returns:
        --------
        float : Accuracy score
        """
        predictions = self.predict(X_test)
        return np.mean(predictions == y_test)

print("KNNClassifier class created successfully!")


### Step 3: Split Data and Apply Feature Scaling


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

# Feature scaling is CRITICAL for KNN!
scaler = StandardScaler() #z-score = (x - mean) / std
X_train_scaled = scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nBefore scaling (first 3 training samples):")
print(X_train[:3])
print("\nAfter scaling (first 3 training samples):")
print(X_train_scaled[:3])


In [ ]:
scaler.mean_, scaler.scale_

### Step 4: Train and Test the Brute-Force KNN Model


In [ ]:
# Create and train the model
knn = KNNClassifier(k=5, distance_metric='euclidean')
knn.fit(X_train_scaled, y_train)

# Measure prediction time
start_time = time.time()
y_pred = knn.predict(X_test_scaled)
prediction_time = time.time() - start_time

# Calculate accuracy
accuracy = knn.score(X_test_scaled, y_test)

print(f"Number of neighbors (k): {knn.k}")
print(f"Distance metric: {knn.distance_metric}")
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Prediction time: {prediction_time:.4f} seconds")
print("\nSample predictions (first 10):")
print(f"Predicted: {y_pred[:10]}")
print(f"Actual:    {y_test[:10]}")


### Step 5: Evaluate with Confusion Matrix


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris.target_names)
disp.plot(cmap='Blues', ax=ax)
plt.title('Confusion Matrix - KNN Classifier (From Scratch)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Confusion Matrix:")
print(cm)


### Step 6: Visualize Decision Boundaries (2D)


In [ ]:
# Use only first 2 features for visualization
X_2d = X[:, :2]
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y, test_size=0.3, random_state=42, stratify=y
)

# Scale the 2D features
scaler_2d = StandardScaler()
X_train_2d_scaled = scaler_2d.fit_transform(X_train_2d)
X_test_2d_scaled = scaler_2d.transform(X_test_2d)

# Train KNN on 2D data
knn_2d = KNNClassifier(k=5, distance_metric='euclidean')
knn_2d.fit(X_train_2d_scaled, y_train_2d)

# Create mesh for decision boundary
h = 0.02  # step size in the mesh
x_min, x_max = X_train_2d_scaled[:, 0].min() - 1, X_train_2d_scaled[:, 0].max() + 1
y_min, y_max = X_train_2d_scaled[:, 1].min() - 1, X_train_2d_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Predict for each point in the mesh
Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot decision boundaries
fig, ax = plt.subplots(figsize=(12, 8))
contourf = ax.contourf(xx, yy, Z, alpha=0.4, cmap='viridis')
scatter = ax.scatter(X_train_2d_scaled[:, 0], X_train_2d_scaled[:, 1],
                     c=y_train_2d, cmap='viridis', edgecolor='black', s=100, alpha=0.8)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Class', fontsize=12)
cbar.set_ticks([0, 1, 2])
cbar.set_ticklabels(iris.target_names)

ax.set_xlabel(iris.feature_names[0], fontsize=12)
ax.set_ylabel(iris.feature_names[1], fontsize=12)
ax.set_title('KNN Decision Boundaries (k=5) - Brute Force', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Calculate accuracy on 2D data
accuracy_2d = knn_2d.score(X_test_2d_scaled, y_test_2d)
print(f"Accuracy with only 2 features: {accuracy_2d:.4f} ({accuracy_2d*100:.2f}%)")


## 2.2 K-D Tree Implementation

### What is a K-D Tree?

A **K-D Tree (K-Dimensional Tree)** is a space-partitioning data structure that organizes points in k-dimensional space. It's particularly useful for efficient nearest neighbor searches.

#### How K-D Trees Work:

1. **Construction**: 
   - Choose a dimension (typically alternating through dimensions)
   - Find the median point along that dimension
   - Split the space at that point
   - Recursively apply to left and right subspaces

2. **Search**:
   - Start at the root
   - Navigate down the tree based on comparison with split values
   - Backtrack to check if there could be closer points in other branches

#### Complexity:
- **Construction**: O(n log n)
- **Search**: O(log n) average case (but O(n) worst case in high dimensions)
- **Space**: O(n)

#### When to Use K-D Trees:
- Low to moderate dimensional data (< 20 dimensions)
- Multiple queries on the same dataset
- When query time is more important than construction time

The K-D tree becomes less effective in high dimensions due to the **curse of dimensionality** - the tree depth approaches n, making it no better than brute force.


### K-D Tree KNN Implementation using scipy


In [ ]:
from scipy.spatial import KDTree


class KNNClassifierKDTree:
    """
    K-Nearest Neighbors Classifier using K-D Tree for efficient neighbor search.
    """

    def __init__(self, k=3):
        """
        Initialize KNN Classifier with K-D Tree.

        Parameters:
        -----------
        k : int
            Number of nearest neighbors to consider
        """
        self.k = k
        self.kdtree = None

    def fit(self, X_train, y_train):
        """
        Fit the model by building a K-D Tree from training data.

        Parameters:
        -----------
        X_train : array-like, shape (n_samples, n_features)
            Training data
        y_train : array-like, shape (n_samples,)
            Target labels
        """
        self.X_train = np.array(X_train)
        self.y_train = np.array(y_train)

        # Build K-D Tree (this is where the magic happens!)
        self.kdtree = KDTree(self.X_train)
        return self

    def predict_single(self, x):
        """
        Predict the class for a single sample using K-D Tree.

        Parameters:
        -----------
        x : array-like, shape (n_features,)
            Test sample

        Returns:
        --------
        int : Predicted class label
        """
        # Query K-D Tree for k nearest neighbors
        # Returns: distances, indices
        distances, indices = self.kdtree.query(x, k=self.k)

        # Get labels of k nearest neighbors
        k_nearest_labels = self.y_train[indices]

        # Return most common label (majority vote)
        most_common = Counter(k_nearest_labels).most_common(1)
        return most_common[0][0]

    def predict(self, X_test):
        """
        Predict classes for multiple samples.

        Parameters:
        -----------
        X_test : array-like, shape (n_samples, n_features)
            Test data

        Returns:
        --------
        array : Predicted class labels
        """
        X_test = np.array(X_test)
        return np.array([self.predict_single(x) for x in X_test])

    def score(self, X_test, y_test):
        """
        Calculate accuracy score.

        Parameters:
        -----------
        X_test : array-like, shape (n_samples, n_features)
            Test data
        y_test : array-like, shape (n_samples,)
            True labels

        Returns:
        --------
        float : Accuracy score
        """
        predictions = self.predict(X_test)
        return np.mean(predictions == y_test)

print("KNNClassifierKDTree class created successfully!")


### Performance Comparison: Brute Force vs K-D Tree


In [ ]:
# Train K-D Tree KNN
knn_kdtree = KNNClassifierKDTree(k=5)

# Measure training time (K-D tree construction)
start_time = time.time()
knn_kdtree.fit(X_train_scaled, y_train)
kdtree_train_time = time.time() - start_time

# Measure prediction time
start_time = time.time()
y_pred_kdtree = knn_kdtree.predict(X_test_scaled)
kdtree_pred_time = time.time() - start_time

# Calculate accuracy
accuracy_kdtree = knn_kdtree.score(X_test_scaled, y_test)

# Compare with brute-force
print("=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)
print(f"\n{'Method':<20} {'Train Time (s)':<15} {'Predict Time (s)':<18} {'Accuracy':<10}")
print("-" * 60)

# Measure brute-force times
start_time = time.time()
knn_brute = KNNClassifier(k=5)
knn_brute.fit(X_train_scaled, y_train)
brute_train_time = time.time() - start_time

start_time = time.time()
y_pred_brute = knn_brute.predict(X_test_scaled)
brute_pred_time = time.time() - start_time

accuracy_brute = knn_brute.score(X_test_scaled, y_test)

print(f"{'Brute-Force':<20} {brute_train_time:<15.6f} {brute_pred_time:<18.6f} {accuracy_brute:<10.4f}")
print(f"{'K-D Tree':<20} {kdtree_train_time:<15.6f} {kdtree_pred_time:<18.6f} {accuracy_kdtree:<10.4f}")
print("-" * 60)

# Speedup
speedup = brute_pred_time / kdtree_pred_time if kdtree_pred_time > 0 else float('inf')
print(f"\nK-D Tree Prediction Speedup: {speedup:.2f}x faster")
print("\nNote: K-D Tree has construction overhead but faster predictions.")
print("For small datasets like Iris, the difference may not be significant.")


### Visualizing K-D Tree Partitioning (2D)


In [ ]:
# Train K-D Tree KNN on 2D data
knn_kdtree_2d = KNNClassifierKDTree(k=5)
knn_kdtree_2d.fit(X_train_2d_scaled, y_train_2d)

# Create decision boundary visualization
h = 0.02
x_min, x_max = X_train_2d_scaled[:, 0].min() - 1, X_train_2d_scaled[:, 0].max() + 1
y_min, y_max = X_train_2d_scaled[:, 1].min() - 1, X_train_2d_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

Z = knn_kdtree_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
contourf = ax.contourf(xx, yy, Z, alpha=0.4, cmap='viridis')
scatter = ax.scatter(X_train_2d_scaled[:, 0], X_train_2d_scaled[:, 1],
                     c=y_train_2d, cmap='viridis', edgecolor='black', s=100, alpha=0.8)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Class', fontsize=12)
cbar.set_ticks([0, 1, 2])
cbar.set_ticklabels(iris.target_names)

ax.set_xlabel(iris.feature_names[0], fontsize=12)
ax.set_ylabel(iris.feature_names[1], fontsize=12)
ax.set_title('KNN Decision Boundaries (k=5) - K-D Tree', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

accuracy_kdtree_2d = knn_kdtree_2d.score(X_test_2d_scaled, y_test_2d)
print(f"K-D Tree Accuracy with 2 features: {accuracy_kdtree_2d:.4f} ({accuracy_kdtree_2d*100:.2f}%)")


# 3. Scikit-learn Implementation

Now that we understand how KNN works internally, let's use scikit-learn's optimized implementation for both classification and regression tasks.

## 3.1 KNN Classification with Scikit-learn

We'll use the same Iris dataset and explore comprehensive evaluation metrics.


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.neighbors import KNeighborsClassifier

# Create and train the model
knn_sklearn = KNeighborsClassifier(n_neighbors=5, metric='euclidean', algorithm='auto')
knn_sklearn.fit(X_train_scaled, y_train)

# Make predictions
y_pred_sklearn = knn_sklearn.predict(X_test_scaled)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred_sklearn)
precision = precision_score(y_test, y_pred_sklearn, average='weighted')
recall = recall_score(y_test, y_pred_sklearn, average='weighted')
f1 = f1_score(y_test, y_pred_sklearn, average='weighted')

print("=" * 60)
print("SCIKIT-LEARN KNN CLASSIFICATION RESULTS")
print("=" * 60)
print(f"\nAccuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

print("\n" + "=" * 60)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred_sklearn, target_names=iris.target_names))


In [ ]:
# Visualize confusion matrix
cm_sklearn = confusion_matrix(y_test, y_pred_sklearn)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_sklearn, display_labels=iris.target_names)
disp.plot(cmap='Blues', ax=ax, values_format='d')
plt.title('Confusion Matrix - Scikit-learn KNN', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Print confusion matrix interpretation
print("\nConfusion Matrix Interpretation:")
print(f"Total correct predictions: {np.trace(cm_sklearn)}")
print(f"Total incorrect predictions: {cm_sklearn.sum() - np.trace(cm_sklearn)}")


### Comparing Different k Values


In [ ]:
# Test different k values
k_values = range(1, 31)
train_scores = []
test_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)

    # Training accuracy
    train_scores.append(knn.score(X_train_scaled, y_train))

    # Test accuracy
    test_scores.append(knn.score(X_test_scaled, y_test))

# Plot the results
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(k_values, train_scores, 'o-', label='Training Accuracy', linewidth=2, markersize=6)
ax.plot(k_values, test_scores, 's-', label='Test Accuracy', linewidth=2, markersize=6)
ax.set_xlabel('Number of Neighbors (k)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('KNN Classification: Accuracy vs. k', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Find optimal k
optimal_k = k_values[np.argmax(test_scores)]
best_test_accuracy = max(test_scores)
print(f"Optimal k: {optimal_k}")
print(f"Best test accuracy: {best_test_accuracy:.4f} ({best_test_accuracy*100:.2f}%)")
print("\nNote: As k increases, the model becomes simpler (higher bias, lower variance)")


## 3.2 KNN Regression with Scikit-learn

KNN can also be used for regression tasks. Instead of majority voting, it predicts the average (or weighted average) of the k nearest neighbors' values.

We'll use the California Housing dataset to demonstrate KNN regression.


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor

# Load California Housing dataset
housing = fetch_california_housing()
X_housing = housing.data
y_housing = housing.target

print("California Housing Dataset Loaded")
print(f"Features shape: {X_housing.shape}")
print(f"Target shape: {y_housing.shape}")
print(f"\nFeature names: {housing.feature_names}")
print("Target: Median house value (in $100,000s)")
print(f"\nFirst 5 samples (features):\n{X_housing[:5]}")
print(f"First 5 targets: {y_housing[:5]}")


In [ ]:
# Use a subset for faster computation (KNN can be slow on large datasets)
# In practice, you might want to use the full dataset
from sklearn.utils import resample

X_housing_sample, y_housing_sample = resample(X_housing, y_housing, n_samples=5000, random_state=42)

# Split the data
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    X_housing_sample, y_housing_sample, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train_h.shape[0]}")
print(f"Test set size: {X_test_h.shape[0]}")

# Feature scaling is ESSENTIAL for KNN regression!
scaler_h = StandardScaler()
X_train_h_scaled = scaler_h.fit_transform(X_train_h)
X_test_h_scaled = scaler_h.transform(X_test_h)

print("\nData split and scaled successfully!")


In [ ]:
# Train KNN Regressor
knn_reg = KNeighborsRegressor(n_neighbors=5, weights='uniform')
knn_reg.fit(X_train_h_scaled, y_train_h)

# Make predictions
y_pred_h = knn_reg.predict(X_test_h_scaled)

# Calculate regression metrics
mse = mean_squared_error(y_test_h, y_pred_h)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_h, y_pred_h)
r2 = r2_score(y_test_h, y_pred_h)

print("=" * 60)
print("KNN REGRESSION RESULTS")
print("=" * 60)
print(f"\nMean Squared Error (MSE):  {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R² Score: {r2:.4f}")

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)
print(f"On average, predictions are off by ${mae*100000:.2f}")
print(f"The model explains {r2*100:.2f}% of the variance in house prices")


In [ ]:
# Visualize predictions vs actual values
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot: Predicted vs Actual
axes[0].scatter(y_test_h, y_pred_h, alpha=0.5, s=30)
axes[0].plot([y_test_h.min(), y_test_h.max()], [y_test_h.min(), y_test_h.max()],
             'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Values ($100k)', fontsize=12)
axes[0].set_ylabel('Predicted Values ($100k)', fontsize=12)
axes[0].set_title('KNN Regression: Predicted vs Actual', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residual plot
residuals = y_test_h - y_pred_h
axes[1].scatter(y_pred_h, residuals, alpha=0.5, s=30)
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Values ($100k)', fontsize=12)
axes[1].set_ylabel('Residuals', fontsize=12)
axes[1].set_title('Residual Plot', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Good residual plots show:")
print("1. Random scatter around zero (no pattern)")
print("2. Constant variance across all predicted values")
print("3. Most points close to the zero line")


In [ ]:
# Compare different k values for regression
k_values_reg = range(1, 31)
train_r2_scores = []
test_r2_scores = []
train_rmse_scores = []
test_rmse_scores = []

for k in k_values_reg:
    knn_r = KNeighborsRegressor(n_neighbors=k)
    knn_r.fit(X_train_h_scaled, y_train_h)

    # Training R² score
    train_r2_scores.append(knn_r.score(X_train_h_scaled, y_train_h))

    # Test R² score
    test_r2_scores.append(knn_r.score(X_test_h_scaled, y_test_h))

    # Training RMSE
    y_pred_train_k = knn_r.predict(X_train_h_scaled)
    train_rmse_scores.append(np.sqrt(mean_squared_error(y_train_h, y_pred_train_k)))

    # Test RMSE
    y_pred_k = knn_r.predict(X_test_h_scaled)
    test_rmse_scores.append(np.sqrt(mean_squared_error(y_test_h, y_pred_k)))

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# R² scores
axes[0].plot(k_values_reg, train_r2_scores, 'o-', label='Training R²', linewidth=2, markersize=6)
axes[0].plot(k_values_reg, test_r2_scores, 's-', label='Test R²', linewidth=2, markersize=6)
axes[0].set_xlabel('Number of Neighbors (k)', fontsize=12)
axes[0].set_ylabel('R² Score', fontsize=12)
axes[0].set_title('KNN Regression: R² Score vs. k', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# RMSE
axes[1].plot(k_values_reg, train_rmse_scores, 'o-', label='Training RMSE', linewidth=2, markersize=6)
axes[1].plot(k_values_reg, test_rmse_scores, 's-', label='Test RMSE', color='red', linewidth=2, markersize=6)
axes[1].set_xlabel('Number of Neighbors (k)', fontsize=12)
axes[1].set_ylabel('RMSE', fontsize=12)
axes[1].set_title('KNN Regression: RMSE vs. k', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find optimal k
optimal_k_reg = k_values_reg[np.argmax(test_r2_scores)]
best_r2 = max(test_r2_scores)
best_rmse = min(test_rmse_scores)
print(f"Optimal k for regression: {optimal_k_reg}")
print(f"Best test R² score: {best_r2:.4f}")
print(f"Best test RMSE: {best_rmse:.4f}")


# 4. Cross-Validation for Model Evaluation

Cross-validation is a crucial technique for assessing how well a model generalizes to unseen data. It helps us avoid overfitting and provides more reliable performance estimates than a single train-test split.

## Why Cross-Validation?

**Problem with single train-test split:**
- Results can vary significantly depending on which samples end up in training vs. test
- May get lucky (or unlucky) with the split
- Doesn't fully utilize the available data for training

**Solution: Cross-Validation**
- Systematically rotate through different train-test combinations
- More robust performance estimate
- Better use of limited data
- Helps detect overfitting/underfitting

## 4.1 Hold-out Approach (Baseline)

The simplest approach: split data once into train and test sets.


In [ ]:
# We've already done this earlier, but let's formalize it
print("=" * 60)
print("HOLD-OUT APPROACH")
print("=" * 60)

# Split data
X_train_ho, X_test_ho, y_train_ho, y_test_ho = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scale features
scaler_ho = StandardScaler()
X_train_ho_scaled = scaler_ho.fit_transform(X_train_ho)
X_test_ho_scaled = scaler_ho.transform(X_test_ho)

# Train and evaluate
knn_ho = KNeighborsClassifier(n_neighbors=5)
knn_ho.fit(X_train_ho_scaled, y_train_ho)
test_score = knn_ho.score(X_test_ho_scaled, y_test_ho)

print(f"\nTest set size: {len(y_test_ho)} samples ({len(y_test_ho)/len(y)*100:.1f}% of data)")
print(f"Training set size: {len(y_train_ho)} samples ({len(y_train_ho)/len(y)*100:.1f}% of data)")
print(f"\nTest Accuracy: {test_score:.4f}")

print("\n" + "=" * 60)
print("LIMITATIONS OF HOLD-OUT")
print("=" * 60)
print("1. Performance estimate depends on the specific split")
print("2. Some data is never used for training")
print("3. Can give overly optimistic or pessimistic estimates")
print("4. Not ideal for small datasets")


## 4.2 K-Fold Cross-Validation (Manual Implementation)

**How K-Fold Cross-Validation Works:**

1. Split the dataset into K equal-sized folds
2. For each of the K iterations:
   - Use 1 fold as the test set
   - Use remaining K-1 folds as the training set
   - Train and evaluate the model
3. Average the K performance scores

**Advantages:**
- Every sample is used for both training and testing
- More robust performance estimate
- Reduces variance in the estimate

Let's implement this manually to understand the mechanics.


In [ ]:
from sklearn.model_selection import KFold

print("=" * 60)
print("K-FOLD CROSS-VALIDATION (MANUAL)")
print("=" * 60)

# Initialize K-Fold
n_splits = 5
kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Store scores for each fold
fold_scores = []

# Manual iteration through folds
for fold_idx, (train_idx, test_idx) in enumerate(kfold.split(X), 1):
    # Split data
    X_train_fold, X_test_fold = X[train_idx], X[test_idx]
    y_train_fold, y_test_fold = y[train_idx], y[test_idx]

    # Scale features (IMPORTANT: fit scaler only on training data!)
    scaler_fold = StandardScaler()
    X_train_fold_scaled = scaler_fold.fit_transform(X_train_fold)
    X_test_fold_scaled = scaler_fold.transform(X_test_fold)

    # Train and evaluate
    knn_fold = KNeighborsClassifier(n_neighbors=5)
    knn_fold.fit(X_train_fold_scaled, y_train_fold)
    score = knn_fold.score(X_test_fold_scaled, y_test_fold)

    fold_scores.append(score)
    print(f"Fold {fold_idx}: Accuracy = {score:.4f}")

# Calculate statistics
mean_score = np.mean(fold_scores)
std_score = np.std(fold_scores)

print("\n" + "=" * 60)
print("CROSS-VALIDATION RESULTS")
print("=" * 60)
print(f"Mean Accuracy: {mean_score:.4f} (+/- {std_score:.4f})")
print(f"Min Accuracy: {min(fold_scores):.4f}")
print(f"Max Accuracy: {max(fold_scores):.4f}")
print(f"\nCompare with Hold-out: {test_score:.4f}")
print(f"Difference: {abs(mean_score - test_score):.4f}")


## 4.3 Using `cross_validate()`

Sklearn provides `cross_validate()` which automates cross-validation and can compute multiple metrics simultaneously. It also returns both training and test scores.


In [ ]:
from sklearn.model_selection import RepeatedKFold

rkf = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

# Create a pipeline (combines preprocessing and model)
# This ensures scaling is done correctly within each fold
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

# Define multiple scoring metrics
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision_weighted',
    'recall': 'recall_weighted',
    'f1': 'f1_weighted'
}



# Perform cross-validation
cv_results = cross_validate(
    pipeline, X, y,
    cv=5,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1  # Use all available cores
)

# Convert to DataFrame for better visualization
results_df = pd.DataFrame(cv_results)

print("=" * 60)
print("CROSS_VALIDATE() RESULTS")
print("=" * 60)
print("\nDetailed Results:")
print(results_df)

print("\n" + "=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    test_scores = cv_results[f'test_{metric}']
    print(f"\nTest {metric.capitalize()}:")
    print(f"  Mean: {np.mean(test_scores):.4f}")
    print(f"  Std:  {np.std(test_scores):.4f}")
    print(f"  Min:  {np.min(test_scores):.4f}")
    print(f"  Max:  {np.max(test_scores):.4f}")


## 4.4 Using `cross_val_score()`

For quick model evaluation with a single metric, `cross_val_score()` is simpler and more concise.


In [ ]:
from sklearn.model_selection import cross_val_score

print("=" * 60)
print("CROSS_VAL_SCORE() - QUICK EVALUATION")
print("=" * 60)

# Quick cross-validation with single metric
scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy', n_jobs=-1)

print(f"\nFold Scores: {scores}")
print(f"\nMean Accuracy: {scores.mean():.4f}")
print(f"Standard Deviation: {scores.std():.4f}")
print(f"95% Confidence Interval: [{scores.mean() - 1.96*scores.std():.4f}, {scores.mean() + 1.96*scores.std():.4f}]")

# Compare different models quickly
print("\n" + "=" * 60)
print("COMPARING DIFFERENT K VALUES")
print("=" * 60)

k_values_cv = [1, 3, 5, 7, 9, 11, 15, 20]
for k in k_values_cv:
    pipeline_k = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    scores_k = cross_val_score(pipeline_k, X, y, cv=5, scoring='accuracy')
    print(f"k={k:2d}: Mean Accuracy = {scores_k.mean():.4f} (+/- {scores_k.std():.4f})")


# 5. Hyperparameter Tuning with Cross-Validation

**Hyperparameters** are model settings that we choose before training (unlike parameters which are learned during training).

For KNN, key hyperparameters include:
- **n_neighbors (k)**: Number of neighbors to consider
- **weights**: 'uniform' (all neighbors equal) or 'distance' (closer neighbors have more influence)
- **metric**: Distance metric ('euclidean', 'manhattan', 'minkowski', etc.)
- **algorithm**: 'auto', 'ball_tree', 'kd_tree', or 'brute'
- **p**: Power parameter for Minkowski metric

**Goal**: Find the hyperparameter combination that gives the best cross-validation performance.

## 5.1 Manual Grid Search

Grid search exhaustively tries all combinations of hyperparameter values.

**Process:**
1. Define a grid of hyperparameter values to try
2. For each combination:
   - Perform cross-validation
   - Record the mean score
3. Select the combination with the best score


In [ ]:
print("=" * 60)
print("MANUAL GRID SEARCH")
print("=" * 60)

# Define parameter grid
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# Store results
results_list = []

# Nested loops for grid search
total_combinations = (len(param_grid['n_neighbors']) *
                      len(param_grid['weights']) *
                      len(param_grid['metric']))

print(f"\nTesting {total_combinations} combinations...")
print("(This may take a moment...)\n")

combo_idx = 0
for n_neighbors in param_grid['n_neighbors']:
    for weights in param_grid['weights']:
        for metric in param_grid['metric']:
            combo_idx += 1

            # Create pipeline with these hyperparameters
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('knn', KNeighborsClassifier(
                    n_neighbors=n_neighbors,
                    weights=weights,
                    metric=metric
                ))
            ])

            # Cross-validate
            scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
            mean_score = scores.mean()
            std_score = scores.std()

            # Store results
            results_list.append({
                'n_neighbors': n_neighbors,
                'weights': weights,
                'metric': metric,
                'mean_score': mean_score,
                'std_score': std_score
            })

            print(f"[{combo_idx}/{total_combinations}] k={n_neighbors}, weights={weights:8s}, "
                  f"metric={metric:10s} => Accuracy: {mean_score:.4f} (+/- {std_score:.4f})")

# Convert to DataFrame and sort
results_manual_grid = pd.DataFrame(results_list)
results_manual_grid = results_manual_grid.sort_values('mean_score', ascending=False)

print("\n" + "=" * 60)
print("TOP 5 CONFIGURATIONS")
print("=" * 60)
print(results_manual_grid.head())

# Best configuration
best_params = results_manual_grid.iloc[0]
print("\n" + "=" * 60)
print("BEST HYPERPARAMETERS")
print("=" * 60)
print(f"n_neighbors: {int(best_params['n_neighbors'])}")
print(f"weights: {best_params['weights']}")
print(f"metric: {best_params['metric']}")
print(f"Best CV Accuracy: {best_params['mean_score']:.4f} (+/- {best_params['std_score']:.4f})")


In [ ]:
# Visualize grid search results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Heatmap for uniform weights
data_uniform = results_manual_grid[results_manual_grid['weights'] == 'uniform']
pivot_uniform = data_uniform.pivot(index='n_neighbors', columns='metric', values='mean_score')
sns.heatmap(pivot_uniform, annot=True, fmt='.4f', cmap='YlGnBu', ax=axes[0], cbar_kws={'label': 'Accuracy'})
axes[0].set_title('Grid Search Results: Uniform Weights', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Distance Metric', fontsize=12)
axes[0].set_ylabel('Number of Neighbors (k)', fontsize=12)

# Plot 2: Heatmap for distance weights
data_distance = results_manual_grid[results_manual_grid['weights'] == 'distance']
pivot_distance = data_distance.pivot(index='n_neighbors', columns='metric', values='mean_score')
sns.heatmap(pivot_distance, annot=True, fmt='.4f', cmap='YlGnBu', ax=axes[1], cbar_kws={'label': 'Accuracy'})
axes[1].set_title('Grid Search Results: Distance Weights', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Distance Metric', fontsize=12)
axes[1].set_ylabel('Number of Neighbors (k)', fontsize=12)

plt.tight_layout()
plt.show()


## 5.2 Manual Random Search

Random search samples random combinations from the hyperparameter space. It's more efficient than grid search when:
- The hyperparameter space is large
- Some hyperparameters don't significantly affect performance
- You have limited computational resources

**Advantage**: Can explore a wider range of values with fewer evaluations.


In [ ]:
import random

print("=" * 60)
print("MANUAL RANDOM SEARCH")
print("=" * 60)

# Define parameter distributions (wider range than grid search)
param_distributions = {
    'n_neighbors': list(range(1, 31)),  # 1 to 30
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski'],
    'p': [1, 2, 3]  # For minkowski metric
}

# Number of random combinations to try
n_iterations = 20

results_random = []

print(f"\nTesting {n_iterations} random combinations...")
print("(Sampling from a larger parameter space)\n")

# Set seed for reproducibility
random.seed(42)
np.random.seed(42)

for i in range(n_iterations):
    # Randomly sample hyperparameters
    n_neighbors = random.choice(param_distributions['n_neighbors'])
    weights = random.choice(param_distributions['weights'])
    metric = random.choice(param_distributions['metric'])
    p = random.choice(param_distributions['p'])

    # Create pipeline
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(
            n_neighbors=n_neighbors,
            weights=weights,
            metric=metric,
            p=p
        ))
    ])

    # Cross-validate
    scores = cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
    mean_score = scores.mean()
    std_score = scores.std()

    # Store results
    results_random.append({
        'n_neighbors': n_neighbors,
        'weights': weights,
        'metric': metric,
        'p': p,
        'mean_score': mean_score,
        'std_score': std_score
    })

    print(f"[{i+1:2d}/{n_iterations}] k={n_neighbors:2d}, weights={weights:8s}, "
          f"metric={metric:10s}, p={p} => Accuracy: {mean_score:.4f} (+/- {std_score:.4f})")

# Convert to DataFrame and sort
results_random_df = pd.DataFrame(results_random)
results_random_df = results_random_df.sort_values('mean_score', ascending=False)

print("\n" + "=" * 60)
print("TOP 5 RANDOM CONFIGURATIONS")
print("=" * 60)
print(results_random_df.head())

# Best configuration
best_random = results_random_df.iloc[0]
print("\n" + "=" * 60)
print("BEST HYPERPARAMETERS (RANDOM SEARCH)")
print("=" * 60)
print(f"n_neighbors: {int(best_random['n_neighbors'])}")
print(f"weights: {best_random['weights']}")
print(f"metric: {best_random['metric']}")
print(f"p: {int(best_random['p'])}")
print(f"Best CV Accuracy: {best_random['mean_score']:.4f} (+/- {best_random['std_score']:.4f})")

print("\n" + "=" * 60)
print("COMPARISON: GRID vs RANDOM SEARCH")
print("=" * 60)
print(f"Grid Search Best: {best_params['mean_score']:.4f}")
print(f"Random Search Best: {best_random['mean_score']:.4f}")
print(f"Random search tried {n_iterations} combinations vs {total_combinations} in grid search")


## 5.3 GridSearchCV (Automated Grid Search)

Sklearn's `GridSearchCV` automates the grid search process with additional features:
- Parallel processing
- Automatic cross-validation
- Detailed results tracking
- Easy access to best parameters and scores


In [ ]:
from sklearn.model_selection import GridSearchCV

print("=" * 60)
print("GRIDSEARCHCV (AUTOMATED)")
print("=" * 60)

# Create pipeline
pipeline_gs = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# Define parameter grid (note the prefix 'knn__' for pipeline)
param_grid_gs = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan', 'minkowski']
}

# Create GridSearchCV object
grid_search = GridSearchCV(
    estimator=pipeline_gs,
    param_grid=param_grid_gs,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,  # Use all available cores
    verbose=1,
    return_train_score=True
)

# Fit grid search
print("\nRunning GridSearchCV...\n")
grid_search.fit(X, y)

print("\n" + "=" * 60)
print("GRIDSEARCHCV RESULTS")
print("=" * 60)
print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")
print(f"Best Estimator: {grid_search.best_estimator_}")

# Get detailed results
cv_results_gs = pd.DataFrame(grid_search.cv_results_)
cv_results_gs = cv_results_gs.sort_values('rank_test_score')

print("\n" + "=" * 60)
print("TOP 5 CONFIGURATIONS")
print("=" * 60)
print(cv_results_gs[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].head())


In [ ]:
# Visualize GridSearchCV results
fig, ax = plt.subplots(figsize=(14, 8))

# Extract relevant data
n_neighbors_list = [params['knn__n_neighbors'] for params in cv_results_gs['params']]
weights_list = [params['knn__weights'] for params in cv_results_gs['params']]
metric_list = [params['knn__metric'] for params in cv_results_gs['params']]
scores = cv_results_gs['mean_test_score'].values

# Create a scatter plot
for weight in ['uniform', 'distance']:
    for metric in ['euclidean', 'manhattan', 'minkowski']:
        mask = [(w == weight and m == metric) for w, m in zip(weights_list, metric_list)]
        k_vals = [k for k, m in zip(n_neighbors_list, mask) if m]
        scores_filtered = [s for s, m in zip(scores, mask) if m]

        label = f"{weight}, {metric}"
        ax.plot(k_vals, scores_filtered, 'o-', label=label, linewidth=2, markersize=8)

ax.set_xlabel('Number of Neighbors (k)', fontsize=12)
ax.set_ylabel('Mean CV Accuracy', fontsize=12)
ax.set_title('GridSearchCV Results: All Configurations', fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 5.4 RandomizedSearchCV (Automated Random Search)

`RandomizedSearchCV` is particularly useful when:
- The hyperparameter space is very large
- You have limited computational budget
- You want to explore continuous or very large discrete spaces

Unlike `GridSearchCV`, it samples a fixed number of candidates from parameter distributions.


In [ ]:
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

print("=" * 60)
print("RANDOMIZEDSEARCHCV (AUTOMATED)")
print("=" * 60)

# Create pipeline
pipeline_rs = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# Define parameter distributions
param_distributions_rs = {
    'knn__n_neighbors': randint(1, 31),  # Random integers from 1 to 30
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan', 'minkowski', 'chebyshev'],
    'knn__p': [1, 2, 3, 4, 5]  # For Minkowski metric
}

# Create RandomizedSearchCV object
random_search = RandomizedSearchCV(
    estimator=pipeline_rs,
    param_distributions=param_distributions_rs,
    n_iter=30,  # Number of parameter settings sampled
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1,
    random_state=42,
    return_train_score=True
)

# Fit random search
print("\nRunning RandomizedSearchCV...\n")
random_search.fit(X, y)

print("\n" + "=" * 60)
print("RANDOMIZEDSEARCHCV RESULTS")
print("=" * 60)
print(f"\nBest Parameters: {random_search.best_params_}")
print(f"Best CV Score: {random_search.best_score_:.4f}")

# Get detailed results
cv_results_rs = pd.DataFrame(random_search.cv_results_)
cv_results_rs = cv_results_rs.sort_values('rank_test_score')

print("\n" + "=" * 60)
print("TOP 5 CONFIGURATIONS")
print("=" * 60)
print(cv_results_rs[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].head())

print("\n" + "=" * 60)
print("COMPARISON: GRIDSEARCH vs RANDOMIZEDSEARCH")
print("=" * 60)
print(f"GridSearchCV Best Score: {grid_search.best_score_:.4f}")
print(f"RandomizedSearchCV Best Score: {random_search.best_score_:.4f}")
print(f"\nGridSearchCV tested: {len(cv_results_gs)} combinations")
print(f"RandomizedSearchCV tested: {len(cv_results_rs)} combinations")
print("\nRandom search explored a larger space with fewer evaluations!")


In [ ]:
random_search.best_estimator_.predict(X_test_scaled)

In [ ]:
random_search.predict(X_test_scaled)

In [ ]:
random_search.best_params_

In [ ]:
KNeighborsClassifier(n_neighbors=11, weights="distance", p=5, metric="minkowski").fit()

In [ ]:
random_search.predict(X_test_scaled)

In [ ]:
random_search.best_params_

In [ ]:
random_search.best_estimator_.predict(X_test_scaled)

In [ ]:
random_search.predict(X_test_scaled)

In [ ]:
KNeighborsClassifier(n_neighbors=random_search.best_params_["knn__n_neighbors"],
              weights=random_search.best_params_["knn__weights"],
              metric=random_search.best_params_["knn__metric"],
              p=random_search.best_params_["knn__p"]).fit(X_train_scaled, y_train)

# 6. Comprehensive Example: Complete Pipeline

Let's put everything together into a complete, production-ready pipeline that incorporates all best practices we've learned.

## Complete Workflow:
1. Load and explore data
2. Split into train/test sets  
3. Build pipeline with preprocessing
4. Hyperparameter tuning with cross-validation
5. Evaluate final model on test set
6. Analyze results


In [ ]:
print("=" * 70)
print("COMPLETE KNN PIPELINE - BEST PRACTICES")
print("=" * 70)

# Step 1: Load data (using Iris for this example)
print("\n[STEP 1] Loading Data...")
X_full = iris.data
y_full = iris.target
print(f"Dataset shape: {X_full.shape}")
print(f"Classes: {np.unique(y_full)}")

# Step 2: Train/Test Split (IMPORTANT: Split BEFORE any preprocessing!)
print("\n[STEP 2] Splitting Data...")
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)
print(f"Training set: {X_train_final.shape[0]} samples")
print(f"Test set: {X_test_final.shape[0]} samples")

# Step 3: Build Pipeline
print("\n[STEP 3] Building Pipeline...")
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Always scale for KNN!
    ('knn', KNeighborsClassifier())
])
print("Pipeline created: StandardScaler → KNN")

# Step 4: Hyperparameter Tuning with Cross-Validation
print("\n[STEP 4] Hyperparameter Tuning...")
print("Using GridSearchCV with 5-fold cross-validation")

param_grid_final = {
    'knn__n_neighbors': [3, 5, 7, 9, 11],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan']
}

grid_search_final = GridSearchCV(
    estimator=final_pipeline,
    param_grid=param_grid_final,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

# Fit on TRAINING data only
grid_search_final.fit(X_train_final, y_train_final)

print(f"Best parameters found: {grid_search_final.best_params_}")
print(f"Best cross-validation score: {grid_search_final.best_score_:.4f}")

# Step 5: Evaluate on Test Set
print("\n[STEP 5] Final Model Evaluation on Test Set...")
best_model = grid_search_final.best_estimator_
y_pred_final = best_model.predict(X_test_final)

# Calculate all metrics
final_accuracy = accuracy_score(y_test_final, y_pred_final)
final_precision = precision_score(y_test_final, y_pred_final, average='weighted')
final_recall = recall_score(y_test_final, y_pred_final, average='weighted')
final_f1 = f1_score(y_test_final, y_pred_final, average='weighted')

print("\n" + "=" * 70)
print("FINAL MODEL PERFORMANCE")
print("=" * 70)
print(f"Test Accuracy:  {final_accuracy:.4f} ({final_accuracy*100:.2f}%)")
print(f"Test Precision: {final_precision:.4f}")
print(f"Test Recall:    {final_recall:.4f}")
print(f"Test F1-Score:  {final_f1:.4f}")

print("\n" + "=" * 70)
print("DETAILED CLASSIFICATION REPORT")
print("=" * 70)
print(classification_report(y_test_final, y_pred_final, target_names=iris.target_names))


In [ ]:
# Final confusion matrix
cm_final = confusion_matrix(y_test_final, y_pred_final)

fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm_final, display_labels=iris.target_names)
disp.plot(cmap='Greens', ax=ax, values_format='d')
plt.title('Final Model - Confusion Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Performance comparison table
print("\n" + "=" * 70)
print("PERFORMANCE COMPARISON: ALL APPROACHES")
print("=" * 70)

comparison_data = {
    'Approach': [
        'Hold-out (single split)',
        'Manual K-Fold CV',
        'cross_validate()',
        'GridSearchCV',
        'RandomizedSearchCV',
        'Final Tuned Model (Test)'
    ],
    'Accuracy': [
        f"{test_score:.4f}",
        f"{mean_score:.4f}",
        f"{np.mean(cv_results['test_accuracy']):.4f}",
        f"{grid_search.best_score_:.4f}",
        f"{random_search.best_score_:.4f}",
        f"{final_accuracy:.4f}"
    ],
    'Note': [
        'Single evaluation',
        '5-fold average',
        '5-fold with metrics',
        'Best of grid search',
        'Best of random search',
        'Unseen test data'
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))


# 7. Key Takeaways and Best Practices

## Summary of KNN

**Strengths:**
- ✅ Simple and intuitive algorithm
- ✅ No training phase (lazy learning)
- ✅ Non-parametric (no assumptions about data distribution)
- ✅ Naturally handles multi-class classification
- ✅ Can capture complex decision boundaries
- ✅ Works well for small to medium datasets

**Weaknesses:**
- ❌ Computationally expensive at prediction time
- ❌ Requires significant memory to store training data
- ❌ Sensitive to irrelevant features and feature scaling
- ❌ Suffers from curse of dimensionality
- ❌ Poor performance with imbalanced datasets
- ❌ Sensitive to noisy data and outliers

## Best Practices for KNN

### 1. **ALWAYS Scale Your Features**
```python
# Use StandardScaler, MinMaxScaler, or RobustScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use same transformation!
```

### 2. **Use Pipelines**
```python
# Ensures proper preprocessing in cross-validation
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])
```

### 3. **Choose k Wisely**
- Start with $k = \sqrt{n}$ where n is the number of training samples
- Use odd values for binary classification to avoid ties
- Use cross-validation to find optimal k
- Consider: small k → high variance, large k → high bias

### 4. **Feature Selection/Engineering**
- Remove irrelevant features (they add noise to distance calculations)
- Consider PCA or feature selection techniques for high-dimensional data
- Domain knowledge is crucial for feature engineering

### 5. **Handle Imbalanced Data**
- Use `weights='distance'` to give more importance to closer neighbors
- Consider SMOTE or other resampling techniques
- Try different distance metrics

### 6. **Distance Metric Selection**
- **Euclidean**: Default choice, works well for continuous features
- **Manhattan**: More robust to outliers
- **Minkowski**: Generalization (p=1 Manhattan, p=2 Euclidean)
- **Hamming**: For categorical features
- **Cosine**: For text data or when magnitude doesn't matter

### 7. **Algorithm Choice**
- **brute**: Always works, good for small datasets
- **kd_tree**: Fast for low-dimensional data (< 20 features)
- **ball_tree**: Better for high-dimensional data
- **auto**: Let sklearn choose (recommended)

### 8. **Cross-Validation Strategy**
- Never tune hyperparameters on the test set!
- Use nested cross-validation for unbiased performance estimates
- Stratify splits for classification to maintain class balance

### 9. **Performance Optimization**
- Use `n_jobs=-1` for parallel processing
- Consider dimensionality reduction for high-dimensional data
- For very large datasets, consider approximate nearest neighbor algorithms

### 10. **Model Evaluation**
- Use multiple metrics (accuracy, precision, recall, F1)
- Examine confusion matrix for misclassification patterns
- For regression: check residual plots
- Always report confidence intervals (mean ± std from CV)

## When to Use KNN

✅ **Use KNN when:**
- You have small to medium-sized datasets
- Features are mostly continuous and can be scaled
- Decision boundaries are expected to be non-linear
- You need a simple, interpretable baseline model
- Number of features is moderate (< 20-30)
- You need both classification and regression capabilities

❌ **Avoid KNN when:**
- Dataset is very large (slow predictions)
- High-dimensional data (curse of dimensionality)
- Real-time predictions are critical (too slow)
- Features are mostly categorical
- Data is very noisy or has many outliers

## Common Pitfalls to Avoid

1. **Forgetting to scale features** → Dominant features will determine distances
2. **Using k=1** → Highly susceptible to noise and overfitting
3. **Not using cross-validation** → Overly optimistic performance estimates
4. **Including irrelevant features** → Adds noise to distance calculations
5. **Testing on data seen during hyperparameter tuning** → Biased estimates
6. **Not handling imbalanced classes** → Majority class dominates
7. **Using KNN for high-dimensional data without preprocessing** → Poor performance

## Further Reading

- [Scikit-learn KNN Documentation](https://scikit-learn.org/stable/modules/neighbors.html)
- [KNN Algorithm Explained](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm)
- [Curse of Dimensionality](https://en.wikipedia.org/wiki/Curse_of_dimensionality)
- [Cross-Validation Techniques](https://scikit-learn.org/stable/modules/cross_validation.html)


# 8. Homework Assignment

## Wine Quality Classification using KNN

### Objective
Apply everything you've learned about KNN to classify wine quality using the Wine Quality dataset.

### Dataset
Use the **Wine Quality dataset** from sklearn or UCI Machine Learning Repository:
- Binary classification: Good quality (score ≥ 7) vs. Not good quality (score < 7)
- Features: Various chemical properties of wine
- Available via: `sklearn.datasets` or download from UCI

### Tasks

#### Part 1: Data Exploration and Preprocessing 
1. Load the wine quality dataset
2. Explore the dataset:
   - Print shape, feature names, and class distribution
   - Create visualizations (histograms, correlation matrix)
   - Check for missing values and outliers
3. Create binary classification labels (Good/Not Good quality)
4. Split data into train (70%) and test (30%) sets with stratification

#### Part 2: From-Scratch Implementation 
1. Implement a basic KNN classifier from scratch (similar to Section 2.1)
2. Test with different distance metrics (Euclidean, Manhattan)
3. Compare results with k = 3, 5, 7
4. Calculate and report accuracy, precision, recall, and F1-score

#### Part 3: Scikit-learn Implementation with Evaluation 
1. Build a pipeline with StandardScaler and KNeighborsClassifier
2. Compare performance with:
   - Different k values (1, 3, 5, 7, 11, 15, 20)
   - Different weights ('uniform' vs 'distance')
   - Different distance metrics
3. Create visualizations:
   - Accuracy vs. k plot
   - Confusion matrix for best model
   - If possible (2-3 features): decision boundary plot
4. Write a detailed classification report

#### Part 4: Cross-Validation 
1. Implement 5-fold cross-validation manually using a for loop
2. Use `cross_validate()` with multiple scoring metrics
3. Compare CV results with hold-out results
4. Discuss which approach gives more reliable estimates and why

#### Part 5: Hyperparameter Tuning 
1. Use `GridSearchCV` to find optimal hyperparameters:
   - n_neighbors: [3, 5, 7, 9, 11, 13, 15]
   - weights: ['uniform', 'distance']
   - metric: ['euclidean', 'manhattan', 'minkowski']
2. Use `RandomizedSearchCV` with:
   - n_neighbors: randint(1, 31)
   - At least 20 iterations
3. Compare GridSearch vs RandomSearch:
   - Best scores achieved
   - Number of evaluations
   - Computational time
4. Visualize the hyperparameter search results (heatmaps or plots)

#### Part 6: Final Model and Analysis 
1. Train final model with best hyperparameters on full training set
2. Evaluate on test set with all metrics
3. Create a comprehensive performance report including:
   - All evaluation metrics
   - Confusion matrix
   - Comparison table of all approaches
4. **Bonus**: Discuss:
   - Why did certain hyperparameters perform better?
   - What are the limitations of KNN for this problem?
   - How could the model be improved?
   - Compare KNN performance with what you'd expect from other algorithms